# FD002 descriptive EDA (training and validation only)

**Purpose and evidence boundary.** This notebook is a reproducible teaching and diagnostic view of NASA C-MAPSS FD002. It reads only the materialized training and validation splits named in `configs/evaluation/fd002-eda-v1.json`. It never reads the held-out internal test or the official NASA test set, and it never trains or tunes an anomaly detector.

Every figure is descriptive—not final model evidence. FD002 has run-to-failure trajectories but no physical per-cycle anomaly-onset labels. No plot here should be interpreted as causal evidence, ground-truth fault onset, threshold performance, or test performance. Committed cell outputs are intentionally cleared.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from turbofan_anomaly.data.io import FD002_COLUMNS, load_split_csv
from turbofan_anomaly.data.metadata import create_window_metadata
from turbofan_anomaly.data.preprocessing import OP_COLUMNS, SENSOR_COLUMNS, load_preprocessor
from turbofan_anomaly.data.windows import build_window_array
from turbofan_anomaly.evaluation.provenance import (
    find_repository_root,
    resolve_repo_path,
    sha256_file,
    verify_registered_hash,
)


In [ ]:
REPO_ROOT = find_repository_root(Path.cwd())
CONFIG_PATH = resolve_repo_path("configs/evaluation/fd002-eda-v1.json", REPO_ROOT)
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
SEED = int(CONFIG["random_seed"])
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

assert CONFIG["data_access"]["allowed_splits"] == ["train", "validation"]
assert CONFIG["data_access"]["detector_training_allowed"] is False
assert CONFIG["data_access"]["detector_tuning_allowed"] is False

TRAIN_PATH = resolve_repo_path(CONFIG["inputs"]["train_csv"]["path"], REPO_ROOT)
VALIDATION_PATH = resolve_repo_path(CONFIG["inputs"]["validation_csv"]["path"], REPO_ROOT)
OUTPUT_DIR = resolve_repo_path(CONFIG["output"]["report_directory"], REPO_ROOT)
DATA_READY = TRAIN_PATH.is_file() and VALIDATION_PATH.is_file()


def save_figure(fig: plt.Figure, name: str) -> None:
    """Save a versioned descriptive figure only during an intentional execution."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        OUTPUT_DIR / f"{name}.{CONFIG['output']['figure_format']}",
        dpi=int(CONFIG["output"]["figure_dpi"]),
        bbox_inches="tight",
    )


print(f"Repository: {REPO_ROOT}")
print(f"Config: {CONFIG_PATH.relative_to(REPO_ROOT).as_posix()}")
print(f"Training/validation inputs available: {DATA_READY}")
if not DATA_READY:
    print("Execution pending: required LFS/materialized split inputs are absent; nothing will be downloaded or fabricated.")


In [ ]:
train = validation = None
if DATA_READY:
    observed_train_hash = sha256_file(TRAIN_PATH)
    observed_validation_hash = sha256_file(VALIDATION_PATH)
    assert observed_train_hash == CONFIG["inputs"]["train_csv"]["sha256"]
    assert observed_validation_hash == CONFIG["inputs"]["validation_csv"]["sha256"]
    train = load_split_csv(TRAIN_PATH)
    validation = load_split_csv(VALIDATION_PATH)
    assert list(train.columns) == FD002_COLUMNS
    assert list(validation.columns) == FD002_COLUMNS
    print("Loaded the two explicitly allowed splits with registered hashes.")
else:
    print("Skipped loading: training and/or validation CSV is absent.")


## FD002 schema

Each row is one engine cycle: engine ID, cycle number, three operating settings, and 21 sensor measurements. The schema does not contain a physical anomaly-onset label, a failure-mode label, or an operational alert threshold.


In [ ]:
schema = pd.DataFrame(
    {
        "column": FD002_COLUMNS,
        "role": (
            ["engine identity", "chronological cycle"]
            + ["operating setting"] * 3
            + ["sensor measurement"] * 21
        ),
    }
)
display(schema)


## Rows, engines, cycles, and split counts

The frozen manifest declares 156 training, 52 validation, and 52 held-out internal-test engines. This notebook verifies only the two allowed materialized populations; it does not open the held-out population.


In [ ]:
def split_summary(frame: pd.DataFrame, split_name: str) -> dict[str, int | str]:
    life = frame.groupby("engine", sort=True)["cycle"].max()
    return {
        "split": split_name,
        "rows": int(len(frame)),
        "engines": int(frame["engine"].nunique()),
        "minimum_life_cycles": int(life.min()),
        "median_life_cycles": int(life.median()),
        "maximum_life_cycles": int(life.max()),
    }


if DATA_READY:
    count_table = pd.DataFrame(
        [split_summary(train, "train"), split_summary(validation, "validation")]
    )
    display(count_table)
    assert count_table.set_index("split").loc["train", "engines"] == 156
    assert count_table.set_index("split").loc["validation", "engines"] == 52
else:
    print("Count verification pending with the split CSVs.")


## Engine-life distribution

Engine maximum cycle is an observed trajectory length, not a fault-onset label. The distributions below describe only training and validation engines.


In [ ]:
if DATA_READY:
    life = pd.concat(
        [
            train.groupby("engine")["cycle"].max().rename("max_cycle").to_frame().assign(split="train"),
            validation.groupby("engine")["cycle"].max().rename("max_cycle").to_frame().assign(split="validation"),
        ]
    ).reset_index()
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(data=life, x="max_cycle", hue="split", element="step", stat="count", common_norm=False, ax=ax)
    ax.set_title("Descriptive only — engine-life distribution")
    ax.set_xlabel("Observed maximum cycle")
    save_figure(fig, "engine_life_distribution")
    plt.show()
else:
    print("Engine-life plot pending.")


## Missing, non-finite, constant, and near-constant sensors

This quality check is split-specific. A low-variance sensor may still carry regime information, so the table is diagnostic rather than an automatic feature-removal rule.


In [ ]:
def sensor_quality(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    values = frame.loc[:, list(SENSOR_COLUMNS)].apply(pd.to_numeric, errors="coerce")
    variance = values.var(ddof=0)
    return pd.DataFrame(
        {
            "split": split_name,
            "sensor": list(SENSOR_COLUMNS),
            "missing": values.isna().sum().to_numpy(),
            "non_finite": (~np.isfinite(values.to_numpy(dtype=float))).sum(axis=0),
            "unique_values": values.nunique(dropna=True).to_numpy(),
            "variance": variance.to_numpy(),
            "constant": (values.nunique(dropna=True) <= 1).to_numpy(),
            "near_constant": (variance <= CONFIG["analysis"]["near_constant_variance_threshold"]).to_numpy(),
        }
    )


if DATA_READY:
    quality = pd.concat(
        [sensor_quality(train, "train"), sensor_quality(validation, "validation")],
        ignore_index=True,
    )
    display(quality)
else:
    print("Sensor-quality checks pending.")


## Operating-setting distributions

The three settings are legitimate operating context. Distribution differences can create apparent sensor anomalies, which motivates—but does not by itself validate—condition-aware preprocessing.


In [ ]:
if DATA_READY:
    settings = pd.concat(
        [train.assign(split="train"), validation.assign(split="validation")],
        ignore_index=True,
    )
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for column, ax in zip(OP_COLUMNS, axes, strict=True):
        sns.histplot(data=settings, x=column, hue="split", element="step", stat="density", common_norm=False, ax=ax)
        ax.set_title(f"Descriptive only — {column}")
    save_figure(fig, "operating_setting_distributions")
    plt.show()
else:
    print("Operating-setting plots pending.")


## K=4/6/8 selection evidence summary

This section reads the frozen preprocessing report; it does not refit K-Means. K=6 was selected using validation silhouette, occupancy, and stability evidence. The clusters are operating-setting groups, not proven physical regimes.


In [ ]:
K_REPORT = resolve_repo_path(CONFIG["inputs"]["k_selection_report"]["path"], REPO_ROOT)
if K_REPORT.is_file():
    k_hash = verify_registered_hash(K_REPORT, CONFIG["inputs"]["k_selection_report"]["sha256"])
    k_evidence = pd.read_csv(K_REPORT)
    k_evidence = k_evidence[k_evidence["k"].isin(CONFIG["analysis"]["candidate_k"])].copy()
    display(k_evidence[[
        "k", "validation_silhouette", "stability_ari_mean",
        "stability_ari_min", "validation_min_mode_fraction", "fallback_mode_count"
    ]])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.barplot(data=k_evidence, x="k", y="validation_silhouette", ax=axes[0], color="#4C78A8")
    sns.barplot(data=k_evidence, x="k", y="stability_ari_mean", ax=axes[1], color="#F58518")
    axes[0].set_title("Descriptive evidence — validation silhouette")
    axes[1].set_title("Descriptive evidence — stability ARI")
    save_figure(fig, "k_selection_evidence")
    plt.show()
    print(f"Registered report hash matched as: {k_hash.match_form}")
else:
    print("K-selection report absent; summary pending.")


## Training-versus-validation condition coverage

Coverage is checked with raw operating-setting ranges and the frozen K=6 occupancy report. These are descriptive overlap checks, not detector performance.


In [ ]:
if DATA_READY:
    coverage_rows = []
    for split_name, frame in [("train", train), ("validation", validation)]:
        for column in OP_COLUMNS:
            coverage_rows.append(
                {
                    "split": split_name,
                    "setting": column,
                    "minimum": float(frame[column].min()),
                    "maximum": float(frame[column].max()),
                    "mean": float(frame[column].mean()),
                    "std": float(frame[column].std()),
                }
            )
    display(pd.DataFrame(coverage_rows))
else:
    print("Raw condition-coverage summary pending.")

OCCUPANCY_REPORT = resolve_repo_path(CONFIG["inputs"]["occupancy_report"]["path"], REPO_ROOT)
if OCCUPANCY_REPORT.is_file():
    verify_registered_hash(OCCUPANCY_REPORT, CONFIG["inputs"]["occupancy_report"]["sha256"])
    occupancy = pd.read_csv(OCCUPANCY_REPORT)
    occupancy_k6 = occupancy[occupancy["k"] == CONFIG["analysis"]["selected_k"]].copy()
    display(occupancy_k6)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=occupancy_k6, x="op_mode", y="row_fraction", hue="split", ax=ax)
    ax.set_title("Descriptive evidence — K=6 condition coverage")
    save_figure(fig, "k6_condition_coverage")
    plt.show()
else:
    print("Occupancy report absent; regime-coverage summary pending.")


## Sensor distributions globally and by regime

Global distributions can mix operating conditions. The regime view uses the frozen P1 artifact only when it is already available; the notebook never fits a substitute preprocessor.


In [ ]:
if DATA_READY:
    selected_sensors = CONFIG["analysis"]["trajectory_sensors"][:4]
    long_global = pd.concat(
        [train.assign(split="train"), validation.assign(split="validation")],
        ignore_index=True,
    ).melt(id_vars=["split"], value_vars=selected_sensors, var_name="sensor", value_name="value")
    grid = sns.displot(data=long_global, x="value", hue="split", col="sensor", col_wrap=2, kind="hist", element="step", common_norm=False)
    grid.fig.suptitle("Descriptive only — global sensor distributions", y=1.02)
    save_figure(grid.fig, "sensor_distributions_global")
    plt.show()
else:
    print("Global sensor distributions pending.")

P1_PATH = resolve_repo_path(CONFIG["inputs"]["p1_preprocessor"]["path"], REPO_ROOT)
if DATA_READY and P1_PATH.is_file():
    p1, p1_metadata = load_preprocessor(P1_PATH, expected_split_manifest_id="fd002-primary-v1")
    p1_train = p1.transform(train)
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.boxplot(data=p1_train, x="op_mode", y=selected_sensors[0], showfliers=False, ax=ax)
    ax.set_title(f"Descriptive only — {selected_sensors[0]} after frozen P1, by regime")
    save_figure(fig, "sensor_distribution_by_regime")
    plt.show()
else:
    print("By-regime sensor plot pending because the frozen P1 artifact is absent; no preprocessor was fitted in the notebook.")


## Correlation heatmap (non-causal)

Correlation is descriptive association. It does not establish that one sensor causes degradation, identifies a fault, or should be removed.


In [ ]:
if DATA_READY:
    correlation = train.loc[:, list(SENSOR_COLUMNS)].corr(method=CONFIG["analysis"]["correlation_method"])
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(correlation, cmap="vlag", center=0.0, square=True, ax=ax)
    ax.set_title("Descriptive training correlation — non-causal")
    save_figure(fig, "sensor_correlation_heatmap_noncausal")
    plt.show()
else:
    print("Correlation heatmap pending.")


## Representative normalized-life trajectories

Normalized life is cycle divided by that engine's observed maximum cycle. The plotted training engines are deterministic lifespan-quantile examples, not cherry-picked successes and not physical onset annotations.


In [ ]:
if DATA_READY:
    train_life = train.groupby("engine")["cycle"].max().sort_values()
    representative = []
    for quantile in CONFIG["analysis"]["representative_engine_quantiles"]:
        target = float(train_life.quantile(quantile))
        representative.append(int((train_life - target).abs().idxmin()))
    representative = list(dict.fromkeys(representative))
    trajectories = train[train["engine"].isin(representative)].copy()
    trajectories["normalized_life"] = trajectories["cycle"] / trajectories.groupby("engine")["cycle"].transform("max")
    trajectory_long = trajectories.melt(
        id_vars=["engine", "normalized_life"],
        value_vars=CONFIG["analysis"]["trajectory_sensors"][:4],
        var_name="sensor",
        value_name="value",
    )
    grid = sns.relplot(data=trajectory_long, x="normalized_life", y="value", hue="engine", col="sensor", col_wrap=2, kind="line", facet_kws={"sharey": False})
    grid.fig.suptitle("Descriptive only — representative training trajectories", y=1.02)
    save_figure(grid.fig, "representative_normalized_life_trajectories")
    plt.show()
else:
    print("Representative trajectory plots pending.")


## P0-versus-P1 transformation comparison

P0 and P1 are preprocessing pipelines, not detector architectures. This cell applies already-fitted artifacts to the same deterministic training rows. It does not fit, select, train, tune, threshold, or score a detector.


In [ ]:
P0_PATH = resolve_repo_path(CONFIG["inputs"]["p0_preprocessor"]["path"], REPO_ROOT)
if DATA_READY and P0_PATH.is_file() and P1_PATH.is_file():
    p0, p0_metadata = load_preprocessor(P0_PATH, expected_split_manifest_id="fd002-primary-v1")
    p1, p1_metadata = load_preprocessor(P1_PATH, expected_split_manifest_id="fd002-primary-v1")
    sample = train.sample(n=min(2000, len(train)), random_state=SEED).sort_index()
    p0_sample = p0.transform(sample).assign(pipeline="P0 global")
    p1_sample = p1.transform(sample).assign(pipeline="P1 K=6")
    comparison_sensor = CONFIG["analysis"]["trajectory_sensors"][0]
    comparison = pd.concat([p0_sample, p1_sample], ignore_index=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(data=comparison, x=comparison_sensor, hue="pipeline", element="step", stat="density", common_norm=False, ax=ax)
    ax.set_title(f"Descriptive only — {comparison_sensor}, P0 versus P1")
    save_figure(fig, "p0_p1_transformation_comparison")
    plt.show()
else:
    print("P0-versus-P1 comparison pending because registered preprocessing artifacts are absent; no replacement was fitted.")


## Window construction and shapes

Windows are created through the package API so the notebook does not duplicate pipeline logic. Expected arrays have shape `[N, 30, 21]`; every window remains inside one engine.


In [ ]:
train_metadata = validation_metadata = None
train_windows = validation_windows = None
if DATA_READY:
    train_metadata = create_window_metadata(
        train,
        split="train",
        manifest_id="fd002-primary-v1",
        source_dataset_sha256=CONFIG["dataset"]["source_dataset_sha256"],
        window_size=CONFIG["analysis"]["window_size"],
    )
    validation_metadata = create_window_metadata(
        validation,
        split="validation",
        manifest_id="fd002-primary-v1",
        source_dataset_sha256=CONFIG["dataset"]["source_dataset_sha256"],
        window_size=CONFIG["analysis"]["window_size"],
    )
    train_windows = build_window_array(train, train_metadata)
    validation_windows = build_window_array(validation, validation_metadata)
    shape_table = pd.DataFrame(
        [
            {"split": "train", "rows": len(train), "windows": len(train_metadata), "tensor_shape": str(train_windows.shape)},
            {"split": "validation", "rows": len(validation), "windows": len(validation_metadata), "tensor_shape": str(validation_windows.shape)},
        ]
    )
    display(shape_table)
    assert train_windows.shape == (27583, 30, 21)
    assert validation_windows.shape == (9365, 30, 21)
else:
    print("Window construction pending.")


## Engine-overlap and boundary assertions

These checks cover only the two allowed splits. The held-out internal-test data remains unopened.


In [ ]:
if DATA_READY:
    train_engines = set(train["engine"].astype(int).unique())
    validation_engines = set(validation["engine"].astype(int).unique())
    assert train_engines.isdisjoint(validation_engines)
    for metadata in (train_metadata, validation_metadata):
        assert (metadata["end_cycle"] - metadata["start_cycle"] + 1 == CONFIG["analysis"]["window_size"]).all()
        assert metadata["window_id"].is_unique
        assert set(metadata["label_state"]) == {"unlabeled"}
        assert set(metadata["label_policy_id"]) == {"unassigned"}
    print("PASS: zero train/validation engine overlap; all windows are engine-local, chronological, and unlabeled.")
else:
    print("Leakage and boundary assertions pending.")


## Limitations and links to decisions

- FD002 does not provide physical per-cycle anomaly-onset labels. Late-life policies are evaluation proxies and are not applied in this descriptive notebook.
- K=6 is a validation-selected preprocessing choice; it is not proof of six causal physical regimes.
- Correlations and trajectories are descriptive and non-causal.
- P0/P1 transformation plots require the frozen preprocessing artifacts; absence is reported rather than repaired by notebook fitting.
- No detector is trained or tuned, no threshold/event evaluation is performed, and no final performance claim is produced.
- The held-out internal test and official NASA test remain unopened.
- Execution is pending in this checkout because the required materialized training/validation split files and preprocessing artifacts are absent.

Authority and decisions: [Master Execution Bible v3](../docs/research/MASTER_EXECUTION_BIBLE_V3_RESEARCH_IMPLEMENTATION_2026-08-23.md), [claims ledger](../docs/research/CLAIMS_LEDGER.md), [decision log](../DECISION_LOG.md), and [progress log](../PROGRESS.md).
